# 01 - Schema Verification & EDA

**Support Ticket Triage** - Notebook 1 of 6

This notebook does two things, strictly in order:

1. **Verification** - confirm the columns actually exist and behave as expected.
   Nothing is assumed. Four specific checks decide how later notebooks are built.
2. **EDA** - distributions plus the diagnostics that feed the go/no-go gate in
   Notebook 02.

**Why verification comes first:** this dataset is synthetic. If the labels were
assigned independently of the ticket text, no model will beat chance, and it is
far cheaper to learn that here than in Week 2. The output is `findings.json`,
which every downstream notebook reads instead of re-deriving these numbers.

*Accelerator: **None (CPU)**. Internet: only if the dataset is not mounted under
`/kaggle/input` - see the next cell.*

In [ ]:
import os, re, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
sns.set_theme(style="whitegrid")

WORK = "/kaggle/working"
PLOTS = os.path.join(WORK, "plots")
os.makedirs(PLOTS, exist_ok=True)

def savefig(name):
    """Save to /kaggle/working/plots for the README, and show inline."""
    plt.tight_layout()
    plt.savefig(os.path.join(PLOTS, name + ".png"), dpi=120, bbox_inches="tight")
    plt.show()

print("pandas", pd.__version__)

## Locate the input

**Delete any auto-generated `kagglehub.dataset_load(...)` cell Kaggle inserted.**
It ships with an empty `file_path` and fails with
`ValueError: Unsupported file extension: ''` before reading anything.

This cell finds the CSV itself and handles both mounting styles:

- **Classic** - "Add Input" mounts the dataset read-only under `/kaggle/input`.
  No internet needed.
- **kagglehub** - newer images download on demand instead. **Requires Internet ON**
  in the notebook settings (Settings -> Internet), which needs a phone-verified
  Kaggle account.

In [ ]:
SLUG = "suraj520/customer-support-ticket-dataset"

def find_csvs(base):
    out = []
    for root, _, files in os.walk(base):
        for f in files:
            if f.lower().endswith((".csv", ".tsv")):
                out.append(os.path.join(root, f))
    return out

# Path 1: mounted read-only at /kaggle/input
csv_paths = find_csvs("/kaggle/input") if os.path.isdir("/kaggle/input") else []

# Path 2: kagglehub on-demand download (needs Internet ON)
if not csv_paths:
    print("Nothing under /kaggle/input - falling back to kagglehub download...")
    import kagglehub
    base = kagglehub.dataset_download(SLUG)   # returns a directory, not a file
    print("downloaded to:", base)
    csv_paths = find_csvs(base)

for p in csv_paths:
    print(p)

assert csv_paths, (
    "No CSV found. Either add the dataset via 'Add Input', or enable "
    "Settings -> Internet so kagglehub can download it."
)
CSV = csv_paths[0]
print("\nUsing:", CSV)

In [ ]:
df = pd.read_csv(CSV)
print("shape:", df.shape)
df.head(3)

---
# Part A - Schema verification

No analysis until the schema is confirmed.

In [ ]:
schema = pd.DataFrame({
    "dtype":    df.dtypes.astype(str),
    "nulls":    df.isna().sum(),
    "null_pct": (df.isna().mean() * 100).round(1),
    "nunique":  df.nunique(dropna=True),
})
schema["example"] = [
    df[c].dropna().iloc[0] if df[c].notna().any() else None for c in df.columns
]
schema

In [ ]:
# Top 10 values for every low-cardinality column - this is where surprises show up.
for c in df.columns:
    n = df[c].nunique(dropna=True)
    if n <= 30:
        print(f"\n=== {c}  ({n} unique) ===")
        print(df[c].value_counts(dropna=False).head(10).to_string())

### Resolve column names at runtime

Later notebooks reference these by role, not by literal name, so a renamed column
in a future dataset version fails loudly here instead of silently downstream.

In [ ]:
def pick(*candidates):
    """Return the first candidate present in df, else None."""
    for c in candidates:
        if c in df.columns:
            return c
    return None

COLS = {
    "text":         pick("Ticket Description", "Ticket_Description", "description"),
    "category":     pick("Ticket Type", "Ticket_Type", "category"),
    "subject":      pick("Ticket Subject", "Ticket_Subject"),
    "priority":     pick("Ticket Priority", "Ticket_Priority", "priority"),
    "satisfaction": pick("Customer Satisfaction Rating", "Customer_Satisfaction_Rating"),
    "status":       pick("Ticket Status", "Ticket_Status"),
    "product":      pick("Product Purchased", "Product_Purchased"),
    "id":           pick("Ticket ID", "Ticket_ID"),
}
for k, v in COLS.items():
    print(f"{k:14s} -> {v}")

missing = [k for k in ("text", "category", "priority") if COLS[k] is None]
assert not missing, f"Required column(s) not found: {missing}. Inspect the schema above."

TEXT, CAT, PRI = COLS["text"], COLS["category"], COLS["priority"]

### Check 1 - Template count (duplication-leak risk)

If the descriptions come from a small pool of templates, identical text will land
in both train and test. That inflates every score and is a *leak*, not a result.

A high duplication rate means Notebook 02 must **group-split on the template**
rather than splitting rows independently.

In [ ]:
n_rows   = len(df)
n_unique = df[TEXT].nunique()
dup_rate = 1 - n_unique / n_rows

print(f"rows                 : {n_rows}")
print(f"unique descriptions  : {n_unique}")
print(f"duplication rate     : {dup_rate:.1%}")

vc = df[TEXT].value_counts()
print(f"\nmost repeated description appears {vc.iloc[0]} times")
print(f"descriptions appearing >1 time: {(vc > 1).sum()}")
print("\n--- 3 most common descriptions ---")
for t, n in vc.head(3).items():
    print(f"\n[x{n}] {t[:300]}")

if dup_rate > 0.05:
    print("\n>>> VERDICT: templated text. Notebook 02 MUST group-split on description.")
else:
    print("\n>>> VERDICT: descriptions largely unique. Standard stratified split is safe.")

### Check 2 - The `{product_purchased}` placeholder

Decision, made once, here: **strip the token.** Do *not* substitute the real
`Product Purchased` value. If product correlates with ticket type, substituting it
injects a non-text feature into the text and inflates the category score.

In [ ]:
ph_pattern = r"\{[a-z_]+\}"
has_ph = df[TEXT].astype(str).str.contains(ph_pattern, regex=True)
print(f"rows containing a placeholder: {has_ph.sum()} ({has_ph.mean():.1%})")

found = (df[TEXT].astype(str)
           .str.findall(ph_pattern)
           .explode().dropna().value_counts())
print("\nplaceholder tokens found:")
print(found.to_string() if len(found) else "  (none)")

if has_ph.any():
    print("\n--- example ---")
    print(df.loc[has_ph, TEXT].iloc[0][:400])

### Check 3 - Label cardinality

`num_labels` for Notebooks 04 and 05 comes from **these numbers**, not from the plan.
The original plan assumed 3 priority levels; verify the real count.

In [ ]:
print(f"=== {CAT} ===")
print(df[CAT].value_counts(dropna=False).to_string())
print(f"n_classes = {df[CAT].nunique()}\n")

print(f"=== {PRI} ===")
print(df[PRI].value_counts(dropna=False).to_string())
n_pri = df[PRI].nunique()
print(f"n_classes = {n_pri}")

imb = df[PRI].value_counts()
print(f"\nimbalance ratio (max/min) = {imb.max() / imb.min():.2f}")

if n_pri != 3:
    print(f"\n>>> NOTE: priority has {n_pri} levels, not the 3 originally assumed.")

if COLS["subject"]:
    print(f"\n=== {COLS['subject']} (alternative category target) ===")
    print(f"n_classes = {df[COLS['subject']].nunique()}")
    print(df[COLS["subject"]].value_counts().head(20).to_string())

### Check 4 - Satisfaction rating availability

Expected: non-null only for closed tickets. **The non-null count is the true sample
size for the entire sentiment analysis** - likely much smaller than the row count.

In [ ]:
SAT, STAT = COLS["satisfaction"], COLS["status"]
if SAT:
    n_sat = df[SAT].notna().sum()
    print(f"{SAT}: {n_sat} non-null of {n_rows} ({n_sat / n_rows:.1%})")
    print(df[SAT].value_counts(dropna=False).sort_index().to_string())

    if STAT:
        print("\n--- status x satisfaction-availability ---")
        print(pd.crosstab(df[STAT], df[SAT].notna(),
                          rownames=["status"], colnames=["has_rating"]).to_string())
    print(f"\n>>> Sentiment analysis sample size = {n_sat}")
else:
    n_sat = 0
    print("No satisfaction column found - the sentiment comparison must be reframed.")

---
# Part B - EDA

Distributions first, then the diagnostics that decide the Notebook 02 gate.

In [ ]:
df["_len_char"] = df[TEXT].astype(str).str.len()
df["_len_word"] = df[TEXT].astype(str).str.split().str.len()

print(df[["_len_char", "_len_word"]].describe().round(1).to_string())

# Does max_length=128 truncate a meaningful tail? (word count under-estimates
# wordpiece tokens by ~1.3x, so compare against ~98 words, not 128.)
over = (df["_len_word"] > 98).mean()
print(f"\nrows likely truncated at max_length=128: {over:.1%}")

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df["_len_char"], bins=50, ax=ax[0]).set(title="Ticket length (characters)")
sns.histplot(df["_len_word"], bins=50, ax=ax[1]).set(title="Ticket length (words)")
ax[1].axvline(98, color="crimson", ls="--", label="~128 wordpiece tokens")
ax[1].legend()
savefig("01_length_distributions")

In [ ]:
cols_to_plot = [c for c in [CAT, PRI, SAT] if c]
fig, axes = plt.subplots(1, len(cols_to_plot), figsize=(6 * len(cols_to_plot), 4))
axes = np.atleast_1d(axes)
for ax, c in zip(axes, cols_to_plot):
    order = df[c].value_counts().index
    sns.countplot(data=df, y=c, order=order, ax=ax)
    ax.set_title(f"{c}\n({df[c].nunique()} classes)")
savefig("02_label_distributions")

### Diagnostic 1 - Association between categorical fields

With ~8.5k rows a chi-square p-value is nearly meaningless (everything is
"significant"), so report **Cramer's V**, an effect size on 0-1.

- **V close to 0** - the two fields are independent. If priority is independent of
  both ticket type *and* satisfaction, that is strong evidence it was assigned at
  random.
- **V > 0.1** - some real association worth investigating.

In [ ]:
def cramers_v(a, b):
    ct = pd.crosstab(a, b)
    chi2 = chi2_contingency(ct)[0]
    n = ct.values.sum()
    r, k = ct.shape
    return np.sqrt(chi2 / (n * (min(r, k) - 1))), ct

pairs = [(PRI, CAT)]
if SAT:
    pairs += [(PRI, SAT), (CAT, SAT)]

for a, b in pairs:
    sub = df[[a, b]].dropna()
    v, ct = cramers_v(sub[a], sub[b])
    verdict = "independent - no signal" if v < 0.05 else ("weak" if v < 0.10 else "real association")
    print(f"\n=== {a}  x  {b} ===   Cramers V = {v:.4f}   ({verdict})")
    print(ct.to_string())

### Diagnostic 2 - Top discriminative terms per class

**Read these.** On a real dataset, billing tickets surface "refund", "charge",
"invoice". On randomly-labelled data, the top terms are near-identical across every
class - which is the single clearest visual sign that the task is unlearnable.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

def top_terms(labels, title, n=12):
    mask = labels.notna() & df[TEXT].notna()
    vec = TfidfVectorizer(max_features=5000, stop_words="english", min_df=3)
    X = vec.fit_transform(df.loc[mask, TEXT].astype(str).str.lower())
    vocab = np.array(vec.get_feature_names_out())
    print(f"\n########## Top terms by {title} ##########")
    for cls in sorted(labels[mask].unique(), key=str):
        rows = (labels[mask] == cls).values
        mean = np.asarray(X[rows].mean(axis=0)).ravel()
        print(f"\n[{cls}]  n={rows.sum()}")
        print("  " + ", ".join(vocab[mean.argsort()[::-1][:n]]))

top_terms(df[CAT], CAT)
top_terms(df[PRI], PRI)

### Diagnostic 3 - Urgency-word rate by priority

Two opposite failure modes, and this cell distinguishes them:

- **Concentrated in high priority** - lexical leakage. The model will "succeed" by
  reading the word *urgent*. Document it as a limitation.
- **Flat across levels** - no signal at all. Priority is not predictable from text.

In [ ]:
URGENT = r"\b(urgent|urgently|immediate|immediately|asap|critical|emergency)\b"
df["_urgent"] = df[TEXT].astype(str).str.lower().str.contains(URGENT, regex=True)

rate = (df.groupby(PRI)["_urgent"].agg(["mean", "sum", "size"])
          .rename(columns={"mean": "urgent_rate", "sum": "n_urgent", "size": "n"}))
rate["urgent_rate"] = rate["urgent_rate"].round(4)
print(rate.to_string())

spread = rate["urgent_rate"].max() - rate["urgent_rate"].min()
print(f"\nspread across priority levels = {spread:.4f}")
if spread > 0.10:
    print(">>> LEAKAGE RISK: urgency words concentrate in specific priority levels.")
elif spread < 0.02:
    print(">>> NO SIGNAL: urgency words are flat across priorities.")
else:
    print(">>> Mild association - inconclusive on its own.")

sns.barplot(data=rate.reset_index(), x=PRI, y="urgent_rate").set(
    title="Rate of urgency words by priority")
savefig("03_urgency_by_priority")

---
## Write `findings.json`

Downstream notebooks read these numbers rather than re-deriving them, so the
`num_labels`, split strategy, and sentiment sample size can never drift.

In [ ]:
findings = {
    "csv_path": CSV,
    "n_rows": int(n_rows),
    "columns": COLS,
    "text_col": TEXT,
    "category_col": CAT,
    "priority_col": PRI,
    "n_unique_descriptions": int(n_unique),
    "duplication_rate": round(float(dup_rate), 4),
    "needs_group_split": bool(dup_rate > 0.05),
    "has_placeholder": bool(has_ph.any()),
    "placeholder_rate": round(float(has_ph.mean()), 4),
    "category_classes": sorted(df[CAT].dropna().unique().tolist()),
    "n_category_labels": int(df[CAT].nunique()),
    "priority_classes": sorted(df[PRI].dropna().unique().tolist()),
    "n_priority_labels": int(n_pri),
    "priority_imbalance_ratio": round(float(imb.max() / imb.min()), 3),
    "satisfaction_n": int(n_sat),
    "word_len_p95": float(df["_len_word"].quantile(0.95)),
    "pct_truncated_at_128": round(float(over), 4),
    "urgency_spread": round(float(spread), 4),
}

path = os.path.join(WORK, "findings.json")
with open(path, "w") as f:
    json.dump(findings, f, indent=2, default=str)

print(json.dumps(findings, indent=2, default=str))
print("\nwrote", path)
print("saved plots:", os.listdir(PLOTS))

---
## Read the output before continuing

Do not start Notebook 02 until you can answer all four:

| # | Question | Where |
|---|---|---|
| 1 | Is the text templated? (decides group-split vs stratified split) | Check 1 |
| 2 | How many priority levels? (`num_labels` in Notebook 05) | Check 3 |
| 3 | How many rated tickets? (the real sentiment sample size) | Check 4 |
| 4 | Do the labels relate to the text at all? | Diagnostics 1-3 |

**On question 4** - if Cramer's V is near zero everywhere, the top terms look
identical across classes, and urgency words are flat, then the labels are
independent of the text. Notebook 02's dummy-vs-TF-IDF gate will confirm it
quantitatively, and the project moves to one of the three gate paths in the plan.
That is a legitimate finding, not a failure - but it must be discovered now, not
after two weeks of fine-tuning.

**Before leaving:** *Save Version -> Save & Run All*, so `findings.json` and the
plots become a stable input for Notebook 02.